# Quick Start

This notebook is a first-contact validation path for the Semulo public API. It tests a local conservative medium that produces ground-state-related observables on frustrated lattices. Sites carry state and material conditions; bonds are stateless nearest-neighbor relations that carry no state, strength, or configurable weight.

The medium does not converge to a static assignment. It enters a stable limit-cycle regime. The primary AFM observable is a time-windowed bond-magnitude readout, not a final site snapshot. Instantaneous per-tick and site snapshot readouts are returned as diagnostics alongside the primary measure.

The public API uses integer execution: site states, material rates, and bond displacement readouts are integer-valued; the examples below use 10-bit resolution unless stated otherwise.

This notebook covers eight gates:

1. Confirm API is reachable.
2. Confirm CPU-only runtime.
3. Run square torus bipartite sanity check.
4. Understand why primary windowed readout is `1.0` while instantaneous diagnostic is lower.
5. Run FCC gradient observation.
6. Load FCC `bonds_window.csv`.
7. Recompute the FCC observable from CSV.
8. Compare primary vs diagnostic readouts in a summary table.

Implementation details and update rules are not included. The execution engine is patent-pending and undisclosed.

In [ ]:
import io

import pandas as pd
import requests

BASE = "https://api.semulo.com"

## Gate 1 and 2: Service and Runtime

The `/system` endpoint is part of the validation record. It should report an x86 Linux runtime, 8 vCPU, 16 GiB RAM, and `gpu_available: False`. This records the public runtime as CPU-only for the validation run.

In [ ]:
health = requests.get(f"{BASE}/health", timeout=30).json()
system = requests.get(f"{BASE}/system", timeout=30).json()

print("health:", health)
print()
print("architecture:", system["architecture"])
print("platform:", system["platform"])
print("cpu_count:", system["cpu_count"])
print("memory_limit_gib:", system["memory_limit_gib"])
print("gpu_available:", system["gpu_available"])

assert health["ok"] is True
assert system["ok"] is True
assert system["gpu_available"] is False

## Gate 3 and 4: Bipartite Sanity Check

Before testing the frustrated case, run a bipartite control where the reference observable is known: square torus should reach a primary windowed antiphase fraction of `1.0`.

This cell also demonstrates the most important measurement point in this validation: the medium does not freeze into a static assignment. A square bipartite attractor is a sign-flipping limit cycle where adjacent sites alternate phase each tick. An instantaneous per-tick readout or final site snapshot will undercount the attractor because it catches the system mid-cycle. The primary windowed unsigned bond displacement readout averages over the full cycle window and recovers the correct `1.0` baseline.

The instantaneous diagnostic is returned alongside the primary readout so this gap is always visible and auditable.

In [ ]:
square_payload = {
    "width": 32,
    "height": 32,
    "seed": 0,
    "bits": 10,
    "warmup_ticks": 700,
    "scan_ticks": 300,
    "sample_ticks": 200,
}

sq = requests.post(f"{BASE}/observe/square/2d/torus", json=square_payload, timeout=120).json()
obs = sq["observable"]

print("topology:", sq["topology"])
print("sites:", sq["sites"])
print("bonds:", sq["bonds"])
print()
print("primary_readout:                   ", obs["primary_readout"])
print("antiphase_fraction (primary):      ", obs["antiphase_fraction"])
print("instantaneous_antiphase_mean:      ", obs["instantaneous_antiphase_fraction_mean"])
print("site_snapshot_antiphase_fraction:  ", obs["site_snapshot_antiphase_fraction"])

assert abs(obs["antiphase_fraction"] - 1.0) < 1e-12, "Primary windowed readout should be 1.0 for square torus"
assert obs["instantaneous_antiphase_fraction_mean"] < 0.95, "Instantaneous diagnostic should be below primary for limit-cycle attractor"

## Gate 5: FCC Gradient Observation

The FCC antiferromagnet is a geometrically frustrated lattice with coordination 12. A uniform site material rate leaves the 12-fold symmetry unbroken and produces a stable but sub-reference attractor. A mild z-axis material-rate gradient breaks the symmetry and allows the medium to resolve into the near-reference basin.

For the FCC frustrated case, the comparison reference used here is 2/3 of nearest-neighbor bonds in antiphase relation. This is motivated by antiferromagnetic ordering arguments and has been verified by brute-force enumeration at small lattice sizes; it has not been proven as a strict universal bound for all finite periodic FCC instances.

In [ ]:
fcc_payload = {
    "nx": 8,
    "ny": 8,
    "nz": 8,
    "seed": 0,
    "material_condition": "z_gradient",
}

summary_response = requests.post(f"{BASE}/observe/fcc/torus", json=fcc_payload, timeout=120)
summary_response.raise_for_status()
summary = summary_response.json()

print("version:", summary["version"])
print("sites:", summary["sites"])
print("bonds:", summary["bonds"])
print("cpu_only:", summary["cpu_only"])
print("observable:", summary["observable"])
print("timing:", summary["timing"])

assert summary["ok"] is True
assert summary["sites"] == 256
assert summary["bonds"] == 1536
assert summary["cpu_only"] is True

## Gates 6 and 7: CSV Load and Independent Recompute

The windowed bond CSV exposes per-bond displacement means over the same sample window used by the JSON summary. The mean of `window_antiphase_fraction` across all bonds should match the JSON `antiphase_fraction` exactly for the same payload.

`signed_bond_displacement` preserves orientation from `site_a` to `site_b`. `unsigned_bond_displacement` is the magnitude-only readout used for antiphase classification. Bonds are stateless nearest-neighbor relations; these columns are derived from endpoint site states, not stored on the bond.

In [ ]:
sites_response = requests.post(f"{BASE}/observe/fcc/torus/sites.csv", json=fcc_payload, timeout=120)
window_response = requests.post(f"{BASE}/observe/fcc/torus/bonds_window.csv", json=fcc_payload, timeout=120)

sites_response.raise_for_status()
window_response.raise_for_status()

sites = pd.read_csv(io.StringIO(sites_response.text))
bonds_window = pd.read_csv(io.StringIO(window_response.text))

print(sites.head())
print()
print(bonds_window[[
    "bond",
    "antiphase_count",
    "window_antiphase_fraction",
    "signed_bond_displacement_mean",
    "unsigned_bond_displacement_mean",
]].head())

assert len(sites) == summary["sites"]
assert len(bonds_window) == summary["bonds"]

In [ ]:
json_fraction = summary["observable"]["antiphase_fraction"]
csv_fraction = bonds_window["window_antiphase_fraction"].mean()
difference = abs(json_fraction - csv_fraction)

print("JSON window fraction:", json_fraction)
print("CSV recomputed fraction:", csv_fraction)
print("difference:", difference)

assert difference < 1e-12, "CSV recompute should match JSON summary exactly"

## Gate 8: Primary vs Diagnostic Readout Summary

The table below compares the two audit paths for the FCC observable alongside the square torus diagnostic gap. The JSON and CSV rows are two audit paths to the same primary windowed observable; the instantaneous diagnostic shows what a naive static-snapshot readout would report.

In [ ]:
fcc_obs = summary["observable"]

table = pd.DataFrame([
    {
        "readout": "json_primary_window_fraction (FCC)",
        "value": fcc_obs["antiphase_fraction"],
        "note": "primary windowed observable from JSON summary",
    },
    {
        "readout": "csv_recomputed_window_fraction (FCC)",
        "value": csv_fraction,
        "note": "independently recomputed from bonds_window.csv",
    },
    {
        "readout": "instantaneous_diagnostic_mean (square torus)",
        "value": obs["instantaneous_antiphase_fraction_mean"],
        "note": "per-tick mean; undercounts limit-cycle attractor",
    },
    {
        "readout": "primary_windowed_fraction (square torus)",
        "value": obs["antiphase_fraction"],
        "note": "correct bipartite baseline via windowed readout",
    },
])

table